In [11]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression

# from sklearn.naive_bayes import MultinomialNB
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import LinearSVC

In [12]:
## Loading the dfs
df = pd.read_csv("everything.csv", index_col=0)
df_doc = pd.read_csv('specialists.csv')

In [14]:
# Let's fetch all the possible text data
col = ['label', 'anat_symp_lem', 'symp_lem']
df = df[col]
df = df[(pd.notnull(df['anat_symp_lem'])) & (pd.notnull(df['symp_lem']))]
df['category_id'] = df['label'].factorize()[0]
category_id_df = df[['label', 'category_id']].drop_duplicates().sort_values('category_id')
category_to_id = dict(category_id_df.values)
id_to_category = dict(category_id_df[['category_id', 'label']].values)

In [17]:
#Initialize models and tools
vectorizer = CountVectorizer()
tfidfzer = TfidfTransformer()
lr = LogisticRegression()

In [19]:
#Split train and test data
X_train, X_test, y_train, y_test = train_test_split(df['anat_symp_lem'], df['label'], test_size = 0.3, random_state=42)


In [20]:
#Transform X_train
vectors = vectorizer.fit_transform(X_train)
tfidf = tfidfzer.fit_transform(vectors)

In [24]:
#Train model
model = lr.fit(tfidf,y_train)

In [25]:
from sklearn import metrics
vectors_test = vectorizer.transform(X_test)
pred = lr.predict(vectors_test)

acc_score = metrics.accuracy_score(y_test, pred)
f1_score = metrics.f1_score(y_test, pred, average='macro')

print('Total accuracy classification score: {}'.format(acc_score))
print('Total F1 classification score: {}'.format(f1_score))


Total accuracy classification score: 0.8741496598639455
Total F1 classification score: 0.875514754396742


In [26]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, pred)

array([[85, 15,  2],
       [ 6, 88,  7],
       [ 3,  4, 84]], dtype=int64)

In [34]:
text = ["my head hurts and eyes are watering. ache in temple area."]
# doc = ''.join(list(lr.predict(vectorizer.transform(text))))
result = model.predict(vectorizer.transform(text))[0]

result

'Neurologist'

In [36]:
# Saving model
filename = "CheckApp_LR_Model.pickle"  

with open(filename, 'wb') as file:  
    pickle.dump(model, file)
    
# Saving model
filename = "vectorizer.pickle"  

with open(filename, 'wb') as file:  
    pickle.dump(vectorizer, file)